# 🔍 Hybrid RAG Demo

This notebook walks through the full pipeline:
1. Load & chunk documents
2. Embed and index into ChromaDB
3. Build BM25 index
4. Run hybrid retrieval (Vector + BM25 + RRF)
5. Generate answers with an LLM

> Make sure you've run `pip install -r requirements.txt` and created your `.env` file.

In [ ]:
import sys
sys.path.insert(0, '..')  # Add project root to path

from dotenv import load_dotenv
load_dotenv('../.env')
print('Environment loaded ✅')

## Step 1: Load Documents

In [ ]:
from src.ingestion.document_loader import DocumentLoader

loader = DocumentLoader('../data/sample_docs')
documents = loader.load()

print(f'Loaded {len(documents)} documents')
for doc in documents:
    print(f"  - {doc.metadata['source']} ({len(doc.page_content)} chars)")

## Step 2: Chunk Documents

In [ ]:
from src.ingestion.chunker import RecursiveChunker

chunker = RecursiveChunker(chunk_size=512, overlap=64)
chunks = chunker.split(documents)

print(f'Created {len(chunks)} chunks')
print('\nSample chunk:')
print('-' * 60)
print(chunks[0].text)
print('-' * 60)

## Step 3: Embed and Index (Vector Store)

In [ ]:
from src.ingestion.embedder import Embedder
from src.retrieval.vector_store import VectorStore

embedder = Embedder()  # loads all-MiniLM-L6-v2
vector_store = VectorStore(embedder)
vector_store.add_chunks(chunks)

print(f'VectorStore has {vector_store.count()} documents ✅')

## Step 4: Build BM25 Index

In [ ]:
from src.retrieval.bm25_retriever import BM25Retriever

bm25 = BM25Retriever()
bm25.build(chunks)
print('BM25 index built ✅')

## Step 5: Hybrid Retrieval Comparison

In [ ]:
from src.retrieval.hybrid_retriever import HybridRetriever

query = 'What is BM25 and how does it compare to vector search?'

# Individual retrievers
vec_results = vector_store.search(query, top_k=5)
bm25_results = bm25.search(query, top_k=5)

# Hybrid
retriever = HybridRetriever(vector_store, bm25)
hybrid_results = retriever.retrieve(query, top_k_final=5)

print(f'\n🔵 Vector Search (top 3):')
for i, r in enumerate(vec_results[:3], 1):
    print(f'  {i}. [score={r.score:.4f}] {r.text[:100]}...')

print(f'\n🟡 BM25 Search (top 3):')
for i, r in enumerate(bm25_results[:3], 1):
    print(f'  {i}. [score={r.score:.4f}] {r.text[:100]}...')

print(f'\n🟢 Hybrid RRF (top 3):')
for i, r in enumerate(hybrid_results[:3], 1):
    print(f'  {i}. [rrf={r.score:.6f}] {r.text[:100]}...')

## Step 6: Full RAG Pipeline

In [ ]:
from src.generation.rag_pipeline import RAGPipeline

pipeline = RAGPipeline(retriever)
response = pipeline.run('What are the advantages of hybrid search over pure vector search?')

print('Question:', response.question)
print('\nAnswer:')
print(response.answer)
print('\nSources:')
for s in response.sources:
    print(' -', s)

## Step 7: RRF Score Visualisation

Visualise how RRF combines scores from the two retrievers.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulate RRF score curve
k = 60
ranks = np.arange(1, 21)
rrf_scores = 1.0 / (k + ranks)

plt.figure(figsize=(8, 4))
plt.plot(ranks, rrf_scores, 'o-', color='steelblue', linewidth=2)
plt.xlabel('Rank')
plt.ylabel('RRF Score')
plt.title(f'RRF Score vs Rank (k={k})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/rrf_curve.png', dpi=150)
plt.show()
print('RRF curve saved ✅')